# Symbolize - Proof by induction: $0 + n = n$

`add` is defined over `natrec` recursing on its **second** argument, exactly as
Lean's `Nat.add` does:

$$\mathrm{add}(m, 0) = m \qquad \mathrm{add}(m,\,\mathrm{succ}\,n) = \mathrm{succ}(\mathrm{add}(m, n))$$

So $n + 0 = n$ holds *definitionally* — the recursion fires and both sides
reduce to $n$. But $0 + n$ is **stuck** for a variable $n$: there is nothing
for the recursion to take apart. That statement needs induction, which is what
`natrec` is.

In [1]:
import sys
sys.path.insert(1, "../..")
from symbolize.terms import Var
from symbolize.terms.decl import SET
from symbolize.terms.derive import Argument, Engine, cong_in, eq, hyp, judge, prim, refl
from symbolize.terms.library import N, add, numeral, standard, succ, zero

engine = Engine(standard())          # a private registry
Nat = judge(N, SET, engine=engine)   # the set of naturals

def nat(term, ctx=None):
    """A judgement  term : N."""
    return judge(term, N, ctx or Nat.ctx, engine=engine)

Nat

N set

## The statement

In [2]:
n = hyp("n", Nat)
goal = eq(nat(add(zero, n.term), n.ctx), n)
goal

Id(N, add(0, n), n) set

In [3]:
goal.is_set, goal.repr_typestring()

(True, 'Id(N, add(0, n), n)')

### Why it needs induction

The mirror image is definitional; this one is not.

In [4]:
lhs = nat(add(zero, n.term), n.ctx)          # 0 + n
rhs = nat(add(n.term, zero), n.ctx)          # n + 0
print("0 + n  reduces to:", lhs.run().repr_unicode())
print("n + 0  reduces to:", rhs.run().repr_unicode())
print()
print("0 + n  definitionally n ?", lhs.defeq(n))
print("n + 0  definitionally n ?", rhs.defeq(n))

0 + n  reduces to: natrec((_)ℕ, n, 0, (x, y)succ(y)) : ℕ
n + 0  reduces to: n : ℕ

0 + n  definitionally n ? False
n + 0  definitionally n ? True


In [5]:
# so the mirror image needs no induction at all: refl proves it
rhs.refl

refl(add(n, 0)) : Id(N, add(n, 0), add(n, 0))

## Base case

$0 + 0 = 0$ holds by computation, so reflexivity proves it.

In [6]:
base = refl(nat(zero))
base

refl(0) : Id(N, 0, 0)

In [7]:
base.has_type(eq(nat(add(zero, zero)), nat(zero)))

True

## Step case

Given $x : \mathbb{N}$ and the induction hypothesis $ih : 0 + x = x$, we need
$0 + \mathrm{succ}\,x = \mathrm{succ}\,x$. Since $0 + \mathrm{succ}\,x$
reduces to $\mathrm{succ}(0 + x)$, congruence under `succ` is enough.

In [8]:
x = hyp("x", Nat)
ih = hyp("ih", eq(nat(add(zero, x.term), x.ctx), x))
ih

ih : Id(N, add(0, x), x)

In [9]:
s = Var("s")
step = cong_in(succ(s).abstract(s), ih)   # congruence in the context (s)succ(s)
step

J((x', y, z)Id(N, succ(x'), succ(y)), ih, (x')refl(succ(x'))) : Id(N, succ(add(0, x)), succ(x))

In [10]:
# which is exactly the motive at succ(x)
step.has_type(eq(nat(add(zero, succ(x.term)), x.ctx), nat(succ(x.term), x.ctx)))

True

Discharging both hypotheses gives the step *function* the eliminator wants,
$\forall x.\,(C(x) \Rightarrow C(\mathrm{succ}\,x))$:

In [11]:
f = step.abstract(ih).abstract(x)
f

λ((x)λ((ih)J((x', y, z)Id(N, succ(x'), succ(y)), ih, (x')refl(succ(x'))))) : ∀(N, (x)⟹(Id(N, add(0, x), x), Id(N, succ(add(0, x)), succ(x))))

## Induction

`prim` (N-elimination, [ST] p100) reads the motive $C$ off the step function's
type, checks the base against $C(0)$ and the step against
$\forall x.(C(x) \Rightarrow C(\mathrm{succ}\,x))$ — both up to definitional
equality, which is what let `refl(0)` serve as the base.

In [12]:
body = prim(n, base, f)
body

natrec((x)Id(N, add(0, x), x), n, refl(0), (x, y)J((x', y', z)Id(N, succ(x'), succ(y')), y, (x')refl(succ(x')))) : Id(N, add(0, n), n)

In [13]:
theorem = body.abstract(n)
theorem.type

∀(N, (n)Id(N, add(0, n), n))

In [14]:
theorem

λ((n)natrec((x)Id(N, add(0, x), x), n, refl(0), (x, y)J((x', y', z)Id(N, succ(x'), succ(y')), y, (x')refl(succ(x'))))) : ∀(N, (n)Id(N, add(0, n), n))

## What it rests on

In [15]:
print(theorem.trust_report())

λ(n).(natrec((x)(0 + x) = x, n, refl(0), (x, y)J((x', y', z)succ(x') = succ(y'), y, (x')refl(succ(x'))))) : ∀n.(0 + n) = n depends on no axioms


In [16]:
# the rules built the conclusion; the checker re-verifies the term independently
engine.checker.check(theorem.ctx, theorem.term, theorem.type)
print("re-checked: ok")

re-checked: ok


In [17]:
theorem.repr_context()   # only the empty context: nothing is assumed

'[]'

## The proof is a program

Instantiating at a numeral and normalising runs the induction.

In [18]:
three = nat(numeral(3))
theorem(three)

apply(λ((n)natrec((x)Id(N, add(0, x), x), n, refl(0), (x, y)J((x', y', z)Id(N, succ(x'), succ(y')), y, (x')refl(succ(x'))))), succ(succ(succ(0)))) : Id(N, add(0, succ(succ(succ(0)))), succ(succ(succ(0))))

In [19]:
theorem(three).run()

refl(succ(succ(succ(0)))) : Id(N, add(0, succ(succ(succ(0)))), succ(succ(succ(0))))

In [20]:
for k in (0, 1, 2, 3):
    print(k, "->", theorem(nat(numeral(k))).run().term.repr_typestring())

0 -> refl(0)
1 -> refl(succ(0))
2 -> refl(succ(succ(0)))
3 -> refl(succ(succ(succ(0))))


## As a derivation

In [21]:
Argument(
    [
        Argument([], base, label=r"\mathrm{refl}"),
        Argument([ih], step, discharges=[ih], label=r"\mathrm{cong}"),
    ],
    theorem,
    discharges=[n],
    label=r"\mathrm{natrec}",
)

[n : ℕ] ⋮ 
--------------- (\mathrm{refl})
refl(0) : 0 = 0   [ih : (0 + x) = x] ⋮ ih : (0 + x) = x
------------------------------------------------------------------------------- (\mathrm{cong})
J((x', y, z)succ(x') = succ(y), ih, (x')refl(succ(x'))) : succ(0 + x) = succ(x)
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- (\mathrm{natrec})
λ(n).(natrec((x)(0 + x) = x, n, refl(0), (x, y)J((x', y', z)succ(x') = succ(y'), y, (x')refl(succ(x'))))) : ∀n.(0 + n) = n

## Notes

* Nothing here is asserted: `prim`, `cong_in` and `refl` each build the proof
  *term* as well as the type, so `theorem.term` is a witness the type checker
  can re-verify on its own — which the cell above does.
* `theorem.trust_report()` says *no axioms*. Had we imported
  `Nat.zero_add` from Lean instead (see `symbolize.terms.interop.lean`), the
  statement would be identical but the report would name the import and its
  provenance.
* The same proof read as a program: `natrec` is primitive recursion, the base
  case is the value at zero, and the step is the recursive call. Normalising at
  a numeral evaluates it.